### 1.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

tickers = ["MSFT", "AAPL"]
prices = yf.download(tickers, start="2011-09-01", end="2016-09-02", auto_adjust=False)["Close"]
prices = prices[tickers]
returns = np.log(prices/prices.shift(1)).dropna().to_numpy()

decay = 0.97
mean = np.zeros(2)
cov = np.zeros((2, 2))
for x in returns:
    # the covariance update measures against the mean from before the step so it goes first
    gap = x - mean
    cov = decay*cov + (1 - decay)*np.outer(gap, gap)
    mean = decay*mean + (1 - decay)*x

print("daily returns", len(returns))
print("mean", mean.round(6))
print("covariance")
print(cov)

[*********************100%***********************]  2 of 2 completed

daily returns 1258
mean [0.00125  0.001061]
covariance
[[1.29897109e-04 3.30066088e-05]
 [3.30066088e-05 1.51851715e-04]]


### 2.

In [2]:
caps = np.array([448.77, 575.11])
weights = caps/caps.sum()
portfolio = 1000000.0
position = portfolio*weights

alpha = 0.95
days = 10

loss_mean = -position.dot(mean)
loss_sd = np.sqrt(position.dot(cov).dot(position))
var_one_day = loss_mean + loss_sd*norm.ppf(alpha)
var_ten_day = np.sqrt(days)*var_one_day
capital = 3*var_ten_day

print("weights", weights.round(4))
print("one day VaR", round(var_one_day, 2))
print("ten day VaR", round(var_ten_day, 2))
print("regulatory capital", round(capital, 2))

weights [0.4383 0.5617]
one day VaR 14383.87
ten day VaR 45485.8
regulatory capital 136457.39


### 3.

In [3]:
sd = np.sqrt(np.diag(cov))
corr = cov[0, 1]/(sd[0]*sd[1])

apple = mean[1] - 5*sd[1]
given_mean = mean[0] + corr*sd[0]/sd[1]*(apple - mean[1])
given_sd = sd[0]*np.sqrt(1 - corr**2)
microsoft = given_mean - 5*given_sd
shock = np.array([microsoft, apple])

gap = shock - mean
shock_cov = decay*cov + (1 - decay)*np.outer(gap, gap)
shock_mean = decay*mean + (1 - decay)*shock

print("correlation", round(corr, 4))
print("apple shock", round(apple, 6))
print("microsoft shock", round(microsoft, 6))

correlation 0.235
apple shock -0.060553
microsoft shock -0.067532


### 4. and 5.

In [4]:
paths = 50000
generator = np.random.default_rng(42)

mean1 = np.full(paths, shock_mean[0])
mean2 = np.full(paths, shock_mean[1])
v11 = np.full(paths, shock_cov[0, 0])
v12 = np.full(paths, shock_cov[0, 1])
v22 = np.full(paths, shock_cov[1, 1])
total1 = np.full(paths, shock[0])
total2 = np.full(paths, shock[1])

for k in range(2, days + 1):
    a = np.sqrt(v11)
    b = v12/a
    c = np.sqrt(v22 - b**2)
    z1 = generator.normal(size=paths)
    z2 = generator.normal(size=paths)
    x1 = mean1 + a*z1
    x2 = mean2 + b*z1 + c*z2
    total1 = total1 + x1
    total2 = total2 + x2
    if k < days:
        gap1 = x1 - mean1
        gap2 = x2 - mean2
        v11 = decay*v11 + (1 - decay)*gap1*gap1
        v12 = decay*v12 + (1 - decay)*gap1*gap2
        v22 = decay*v22 + (1 - decay)*gap2*gap2
        mean1 = decay*mean1 + (1 - decay)*x1
        mean2 = decay*mean2 + (1 - decay)*x2

losses = -(position[0]*total1 + position[1]*total2)

### (a) to (d)

In [5]:
ordered = np.sort(losses)
shocked_var = ordered[int(np.ceil(paths*alpha)) - 1]
over_var = (losses > var_ten_day).sum()*100/paths
over_capital = (losses > capital).sum()*100/paths

print("average ten day loss", round(losses.mean(), 2))
print("shocked ten day VaR", round(shocked_var, 2))
print("percent above the initial ten day VaR", round(over_var, 2))
print("percent above the regulatory capital", round(over_capital, 2))

average ten day loss 70988.67
shocked ten day VaR 152045.47
percent above the initial ten day VaR 70.15
percent above the regulatory capital 8.86
